# LangSmith 기초

LangSmith는 LangChain 애플리케이션의 실행 과정을 기록하고 분석하는 관측성(observability) 플랫폼이다. 이 노트북에서는 Gemini 기반 LCEL 체인을 추적한다.


## 설치 및 환경 변수

필요한 패키지를 설치한다.

```bash
pip install langsmith
```

프로젝트 루트의 `.env`에 다음 값을 설정한다.

```dotenv
GEMINI_API_KEY=...
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langchain-gemini-prac
```


API 키가 여러 워크스페이스에 연결되어 있다면 `LANGSMITH_WORKSPACE_ID`도 설정한다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()



print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT", "default"))

LangSmith tracing: true
LangSmith project: langchain-study


## Gemini 체인 만들기

LangChain의 Runnable을 실행하면 별도의 콜백 코드 없이 LangSmith에 트레이스가 기록된다.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 초보자를 가르치는 Python 튜터야. 핵심만 한국어로 설명해줘."),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
chain = prompt | llm | StrOutputParser()

In [3]:
result = chain.invoke({"question": "Python 데코레이터가 뭐야?"})
print(result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


안녕하세요! 파이썬 튜터입니다. 😊

**데코레이터(Decorator)**는 한 마디로 **"기존 함수를 수정하지 않고, 새로운 기능을 얹어주는 도구"**입니다.

이름 그대로 함수를 '장식(Decorate)'해 주는 역할을 해요.

---

### 💡 왜 사용할까요?
여러 함수에 **똑같은 작업**(예: 함수 실행 시간 측정, 로그인 여부 확인, 로그 남기기 등)을 반복해서 써야 할 때, 데코레이터를 쓰면 코드가 훨씬 깨끗해집니다.

---

### 🔍 가장 쉬운 예시

`안녕하세요!`를 출력하는 함수 앞뒤로 "시작"과 "끝"을 알리는 기능을 추가해 볼게요.

```python
# 1. 데코레이터 정의 (기능을 덮어씌울 포장지)
def my_decorator(func):
    def wrapper():
        print("--- 🚀 작업 시작 ---")
        func()  # 원본 함수 실행
        print("--- ✅ 작업 끝 ---")

    return wrapper


# 2. @ 표시를 사용해 함수에 데코레이터 적용
@my_decorator
def hello():
    print("안녕하세요!")


# 3. 함수 실행
hello()
```

**[실행 결과]**
```text
--- 🚀 작업 시작 ---
안녕하세요!
--- ✅ 작업 끝 ---
```

---

### 📌 핵심 요약
1. `hello()` 함수의 내부 코드는 건드리지 않았습니다.
2. 함수 위에 **`@데코레이터이름`**을 붙이기만 하면 추가 기능이 짠! 하고 적용됩니다.
3. **"자주 쓰는 공통 기능을 깔끔하게 재사용하기 위한 기술"**로 기억하시면 됩니다!


## 태그와 메타데이터 추가하기

`config`를 전달하면 LangSmith 화면에서 실행 목적이나 실험 버전을 쉽게 필터링할 수 있다.

In [4]:
result = chain.invoke(
    {"question": "리스트와 튜플의 차이는 뭐야?"},
    config={
        "run_name": "python-basic-question",
        "tags": ["gemini", "basic", "class-demo"],
        "metadata": {"lesson": "01-1", "model_family": "gemini"},
    },
)
print(result)

안녕하세요! 파이썬의 **리스트(List)**와 **튜플(Tuple)**은 여러 데이터를 모아서 관리한다는 점은 같지만, 가장 중요한 차이점은 **"내용을 바꿀 수 있는가?"**입니다.

핵심만 깔끔하게 정리해 드릴게요!

---

### 1. 가장 큰 차이점: 수정 가능 여부

* **리스트 (List)**: 내용 **수정 가능** (추가, 수정, 삭제 OK)
* **튜플 (Tuple)**: 내용 **수정 불가능** (한 번 만들면 변경 불가)

---

### 2. 코드 비교

```python
# 1. 리스트: 대괄호 [] 사용
my_list = [1, 2, 3]
my_list[0] = 99  # 가능! -> [99, 2, 3]으로 바뀜
my_list.append(4)  # 가능! -> [99, 2, 3, 4]로 바뀜

# 2. 튜플: 소괄호 () 사용
my_tuple = (1, 2, 3)
my_tuple[0] = 99  # ❌ 에러 발생! (수정할 수 없음)
```

---

### 3. 한눈에 보는 비교

| 구분 | 리스트 (List) | 튜플 (Tuple) |
| :--- | :--- | :--- |
| **괄호** | 대괄호 `[ ]` | 소괄호 `( )` |
| **수정 여부** | 가능 (Mutable) | **불가능 (Immutable)** |
| **속도/메모리**| 상대적으로 느림 | **더 빠르고 메모리 적게 씀** |

---

### 4. 언제 뭘 써야 할까요?

* **리스트를 쓸 때**: 데이터가 계속 **추가되거나 바뀔 가능성이 있을 때**
  * 예: 장바구니 목록, 학생들의 시험 점수, 게임 스코어
* **튜플을 쓸 때**: 데이터가 **절대 바뀌면 안 되거나, 실수로 바꾸는 걸 방지하고 싶을 때**
  * 예: 지도 좌표(위도, 경도), RGB 색상 코드(255, 255, 255), 생년월일

> **💡 한 줄 요약**
> **"바뀌는 데이터는 리스트 `[]`, 안 바뀌는 데이터는 튜플 `()`!"** 로 기억하세요!


## LangSmith에서 확인하기

[LangSmith](https://smith.langchain.com)에 접속해 `LANGSMITH_PROJECT`에 지정한 프로젝트를 연다.

확인할 항목:

- 체인의 프롬프트 → Gemini 모델 → 출력 파서 실행 순서
- 템플릿 변수가 적용된 최종 프롬프트
- 입력·출력 토큰 수와 각 단계의 소요 시간
- `run_name`, 태그, 메타데이터
- 실패한 실행이 있다면 오류가 발생한 단계

> 환경 변수를 변경했는데 반영되지 않으면 위 환경 설정 셀을 다시 실행한 뒤 커널을 재시작한다.

## 선택적 추적

전체 실행을 항상 기록하지 않고 특정 구간만 추적하려면 `tracing_context`를 사용할 수 있다.

In [ ]:
import langsmith as ls

with ls.tracing_context(enabled=True, project_name="langchain-gemini-selective"):
    result = chain.invoke({"question": "Python 제너레이터가 뭐야?"})

print(result)